<a href="https://colab.research.google.com/github/roaa170/Data-Analysis/blob/main/2015_Flight_Delays_and_Cancellations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd

Data source :- https://www.kaggle.com/datasets/usdot/flight-delays



# load data

In [5]:
flights = pd.read_csv('flights.csv')
airlines = pd.read_csv('airlines.csv')
airports = pd.read_csv('airports.csv')

/tmp/ipython-input-3052370935.py:1: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  flights = pd.read_csv('flights.csv')


# Preprocessing

In [7]:
# Convert datatypes
time_cols = ['SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME']
for col in time_cols:
   flights[col]=pd.to_datetime(flights[col], format='%H%M', errors='coerce')
flights['CANCELLED']=flights['CANCELLED'].astype('bool')
flights['DIVERTED']=flights['DIVERTED'].astype('bool')
flights['FLIGHT_DATE']=pd.to_datetime(flights[['YEAR', 'MONTH', 'DAY']])


In [9]:
# Handle missing values
delay_cols = ['DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY',
              'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']
for col in delay_cols:
   flights[col]=flights[col].fillna(0)

flights['DISTANCE'] = flights['DISTANCE'].fillna(flights['DISTANCE'].median())
flights['SCHEDULED_DEPARTURE'] = flights['SCHEDULED_DEPARTURE'].fillna(flights['SCHEDULED_DEPARTURE'].median())
flights['CANCELLATION_REASON']=flights['CANCELLATION_REASON'].fillna('Not Cancelled')
cancel_reson_map={'A':'Airline/Carrier',
                  'B':'Weather',
                  'C':'National Air System',
                  'D':'Security',
                  'Not Cancelled': 'Not Cancelled'
                  }
flights['CANCELLATION_REASON']=flights['CANCELLATION_REASON'].map(cancel_reson_map)


In [11]:
delay_cols = [
    'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
    'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY'
]
delay_long= flights.melt(
        id_vars=['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'FLIGHT_DATE'],
        value_vars=delay_cols,
        var_name='DELAY_REASON',
        value_name='DELAY_MINUTES'
        )
delay_long=delay_long[delay_long['DELAY_MINUTES']>0]
delay_reason_map = {
    'AIR_SYSTEM_DELAY': 'Air System',
    'SECURITY_DELAY': 'Security',
    'AIRLINE_DELAY': 'Airline',
    'LATE_AIRCRAFT_DELAY': 'Late Aircraft',
    'WEATHER_DELAY': 'Weather'
}
delay_long['DELAY_REASON'] = delay_long['DELAY_REASON'].map(delay_reason_map)

delay_summary=(
    delay_long
    .groupby(['DELAY_REASON'])
    .agg(
        Num_Flights=('DELAY_MINUTES', 'count'),
        Avg_Delay_Minutes=('DELAY_MINUTES', 'mean'),
        Total_Delay_Minutes=('DELAY_MINUTES', 'sum')
    ).sort_values('Total_Delay_Minutes', ascending=False)
    .reset_index()
)
print(delay_long.head())
print(delay_summary)

   AIRLINE ORIGIN_AIRPORT DESTINATION_AIRPORT FLIGHT_DATE DELAY_REASON  \
27      NK            MSP                 FLL  2015-01-01   Air System   
30      NK            PHX                 ORD  2015-01-01   Air System   
50      B6            BQN                 MCO  2015-01-01   Air System   
55      B6            SJU                 BDL  2015-01-01   Air System   
86      AA            DEN                 DFW  2015-01-01   Air System   

    DELAY_MINUTES  
27           25.0  
30           43.0  
50           20.0  
55           17.0  
86           13.0  
    DELAY_REASON  Num_Flights  Avg_Delay_Minutes  Total_Delay_Minutes
0  Late Aircraft       556953          44.818739           24961931.0
1        Airline       570022          35.389785           20172956.0
2     Air System       564826          25.380846           14335762.0
3        Weather        64716          47.905201            3100233.0
4       Security         3484          23.244834              80985.0


# Merge datasets

In [12]:
flights = flights.merge(
        airlines,
        how='left',
        left_on='AIRLINE',
        right_on='IATA_CODE'
    )


In [13]:
flights=flights.merge(
    airports[['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY']],
    how='left',
    left_on='ORIGIN_AIRPORT',
    right_on='IATA_CODE',
    suffixes=('', '_ORIGIN')
)

flights=flights.merge(
    airports[['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY']],
    how='left',
    left_on='DESTINATION_AIRPORT',
    right_on='IATA_CODE',
    suffixes=('', '_ORIGIN')
)


In [14]:
flights = flights.drop(columns=['IATA_CODE_ORIGIN', 'IATA_CODE_DEST'], errors='ignore')
flights = flights.drop_duplicates()


In [15]:
flights['MONTH_NAME']=flights['FLIGHT_DATE'].dt.month_name()
flights['DAY_NAME']=flights['FLIGHT_DATE'].dt.day_name()

In [16]:
flights.columns.to_list()

['YEAR',
 'MONTH',
 'DAY',
 'DAY_OF_WEEK',
 'AIRLINE_x',
 'FLIGHT_NUMBER',
 'TAIL_NUMBER',
 'ORIGIN_AIRPORT',
 'DESTINATION_AIRPORT',
 'SCHEDULED_DEPARTURE',
 'DEPARTURE_TIME',
 'DEPARTURE_DELAY',
 'TAXI_OUT',
 'WHEELS_OFF',
 'SCHEDULED_TIME',
 'ELAPSED_TIME',
 'AIR_TIME',
 'DISTANCE',
 'WHEELS_ON',
 'TAXI_IN',
 'SCHEDULED_ARRIVAL',
 'ARRIVAL_TIME',
 'ARRIVAL_DELAY',
 'DIVERTED',
 'CANCELLED',
 'CANCELLATION_REASON',
 'AIR_SYSTEM_DELAY',
 'SECURITY_DELAY',
 'AIRLINE_DELAY',
 'LATE_AIRCRAFT_DELAY',
 'WEATHER_DELAY',
 'FLIGHT_DATE',
 'IATA_CODE',
 'AIRLINE_y',
 'AIRPORT',
 'CITY',
 'STATE',
 'COUNTRY',
 'AIRPORT_ORIGIN',
 'CITY_ORIGIN',
 'STATE_ORIGIN',
 'COUNTRY_ORIGIN',
 'MONTH_NAME',
 'DAY_NAME']

In [17]:
flights_clean = flights[[
    'AIRLINE_y', 'FLIGHT_NUMBER',
    'ORIGIN_AIRPORT', 'AIRPORT_ORIGIN', 'CITY_ORIGIN', 'STATE_ORIGIN', 'COUNTRY_ORIGIN',
    'DESTINATION_AIRPORT', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY',
    'DEPARTURE_DELAY', 'ARRIVAL_DELAY',
    'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY',
    'CANCELLED', 'CANCELLATION_REASON',
    'DISTANCE', 'ELAPSED_TIME', 'AIR_TIME'
]]


In [18]:
flights.to_csv('flights_clean.csv', index=False)
delay_long.to_csv('delay_long.csv', index=False)
delay_summary.to_csv('delay_summary.csv', index=False)

print("✅ All datasets cleaned, summarized, and saved successfully!")


✅ All datasets cleaned, summarized, and saved successfully!
